### Import Necessary Dependencies

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().resolve().parent))

import matplotlib.pyplot as plt
import seaborn as sns
from src.data_preprocessing import DataPreprocessor

# Initialize preprocessor
preprocessor = DataPreprocessor()
fraud_df, ip_country_df, credit_df = preprocessor.load_data()

# Clean data
fraud_df_clean = preprocessor.clean_fraud_data()
credit_df_clean = preprocessor.clean_credit_data()

### 1. Class Distribution Analysis

In [ ]:
print("\n=== CLASS DISTRIBUTION ===")
print("E-commerce Data:")
fraud_dist = fraud_df_clean['class'].value_counts()
print(f"Non-fraud (0): {fraud_dist[0]} ({fraud_dist[0]/len(fraud_df_clean)*100:.2f}%)")
print(f"Fraud (1): {fraud_dist[1]} ({fraud_dist[1]/len(fraud_df_clean)*100:.2f}%)")

print("\nCredit Card Data:")
credit_dist = credit_df_clean['Class'].value_counts()
print(f"Non-fraud (0): {credit_dist[0]} ({credit_dist[0]/len(credit_df_clean)*100:.2f}%)")
print(f"Fraud (1): {credit_dist[1]} ({credit_dist[1]/len(credit_df_clean)*100:.2f}%)")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].pie(fraud_dist.values, labels=['Non-Fraud', 'Fraud'], autopct='%1.1f%%', colors=['lightblue', 'red'])
axes[0].set_title('E-commerce Fraud Distribution')

axes[1].pie(credit_dist.values, labels=['Non-Fraud', 'Fraud'], autopct='%1.1f%%', colors=['lightblue', 'red'])
axes[1].set_title('Credit Card Fraud Distribution')

plt.tight_layout()
plt.savefig('reports/figures/eda_plots/class_distribution.png', dpi=300, bbox_inches='tight')
plt.show()


### 2. Numerical Features Analysis

In [ ]:
print("\n=== NUMERICAL FEATURES ===")
numerical_cols = ['purchase_value', 'age']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for idx, col in enumerate(numerical_cols):
    # Distribution
    axes[idx, 0].hist(fraud_df_clean[col], bins=30, edgecolor='black', alpha=0.7)
    axes[idx, 0].set_title(f'Distribution of {col}')
    axes[idx, 0].set_xlabel(col)
    axes[idx, 0].set_ylabel('Frequency')
    
    # Box plot by fraud class
    fraud_data = fraud_df_clean[fraud_df_clean['class'] == 1][col]
    non_fraud_data = fraud_df_clean[fraud_df_clean['class'] == 0][col]
    
    axes[idx, 1].boxplot([non_fraud_data, fraud_data], labels=['Non-Fraud', 'Fraud'])
    axes[idx, 1].set_title(f'{col} by Fraud Status')
    axes[idx, 1].set_ylabel(col)

plt.tight_layout()
plt.savefig('reports/figures/eda_plots/numerical_features.png', dpi=300, bbox_inches='tight')
plt.show()

### 3. Categorical Features Analysis

In [ ]:
print("\n=== CATEGORICAL FEATURES ===")
categorical_cols = ['source', 'browser', 'sex']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for idx, col in enumerate(categorical_cols):
    # Calculate fraud rates by category
    fraud_rates = fraud_df_clean.groupby(col)['class'].mean().sort_values(ascending=False)
    
    axes[idx].bar(range(len(fraud_rates)), fraud_rates.values)
    axes[idx].set_title(f'Fraud Rate by {col}')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Fraud Rate')
    axes[idx].set_xticks(range(len(fraud_rates)))
    axes[idx].set_xticklabels(fraud_rates.index, rotation=45)
    
    # Annotate values
    for i, v in enumerate(fraud_rates.values):
        axes[idx].text(i, v + 0.001, f'{v:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('reports/figures/eda_plots/categorical_features.png', dpi=300, bbox_inches='tight')
plt.show()